In [1]:
import pandas as pd
test_df = pd.read_pickle('../data/test_v2.pkl')



In [2]:
test_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 401589 entries, 0 to 401588
Data columns (total 13 columns):
 #   Column                Non-Null Count   Dtype
---  ------                --------------   -----
 0   channelGrouping       401589 non-null  str  
 1   customDimensions      401589 non-null  str  
 2   date                  401589 non-null  int64
 3   device                401589 non-null  str  
 4   fullVisitorId         401589 non-null  str  
 5   geoNetwork            401589 non-null  str  
 6   hits                  401589 non-null  str  
 7   socialEngagementType  401589 non-null  str  
 8   totals                401589 non-null  str  
 9   trafficSource         401589 non-null  str  
 10  visitId               401589 non-null  int64
 11  visitNumber           401589 non-null  int64
 12  visitStartTime        401589 non-null  int64
dtypes: int64(4), str(9)
memory usage: 39.8 MB


In [3]:
test_df.head()

,channelGrouping,customDimensions,date,device,fullVisitorId,geoNetwork,hits,socialEngagementType,totals,trafficSource,visitId,visitNumber,visitStartTime
0,Organic Search,"[{'index': '4', 'value': 'APAC'}]",20180511,"{""browser"": ""Chrome"", ""browserVersion"": ""not a...",7460955084541987166,"{""continent"": ""Asia"", ""subContinent"": ""Souther...","[{'hitNumber': '1', 'time': '0', 'hour': '21',...",Not Socially Engaged,"{""visits"": ""1"", ""hits"": ""4"", ""pageviews"": ""3"",...","{""referralPath"": ""(not set)"", ""campaign"": ""(no...",1526099341,2,1526099341
1,Direct,"[{'index': '4', 'value': 'North America'}]",20180511,"{""browser"": ""Chrome"", ""browserVersion"": ""not a...",460252456180441002,"{""continent"": ""Americas"", ""subContinent"": ""Nor...","[{'hitNumber': '1', 'time': '0', 'hour': '11',...",Not Socially Engaged,"{""visits"": ""1"", ""hits"": ""4"", ""pageviews"": ""3"",...","{""referralPath"": ""(not set)"", ""campaign"": ""(no...",1526064483,166,1526064483
2,Organic Search,"[{'index': '4', 'value': 'North America'}]",20180511,"{""browser"": ""Chrome"", ""browserVersion"": ""not a...",3461808543879602873,"{""continent"": ""Americas"", ""subContinent"": ""Nor...","[{'hitNumber': '1', 'time': '0', 'hour': '12',...",Not Socially Engaged,"{""visits"": ""1"", ""hits"": ""4"", ""pageviews"": ""3"",...","{""referralPath"": ""(not set)"", ""campaign"": ""(no...",1526067157,2,1526067157
3,Direct,"[{'index': '4', 'value': 'North America'}]",20180511,"{""browser"": ""Chrome"", ""browserVersion"": ""not a...",975129477712150630,"{""continent"": ""Americas"", ""subContinent"": ""Nor...","[{'hitNumber': '1', 'time': '0', 'hour': '23',...",Not Socially Engaged,"{""visits"": ""1"", ""hits"": ""5"", ""pageviews"": ""4"",...","{""referralPath"": ""(not set)"", ""campaign"": ""(no...",1526107551,4,1526107551
4,Organic Search,"[{'index': '4', 'value': 'North America'}]",20180511,"{""browser"": ""Internet Explorer"", ""browserVersi...",8381672768065729990,"{""continent"": ""Americas"", ""subContinent"": ""Nor...","[{'hitNumber': '1', 'time': '0', 'hour': '10',...",Not Socially Engaged,"{""visits"": ""1"", ""hits"": ""5"", ""pageviews"": ""4"",...","{""referralPath"": ""(not set)"", ""campaign"": ""(no...",1526060254,1,1526060254


In [4]:
import sys
import pickle
import pandas as pd
import numpy as np
from datetime import datetime
# 1. Cài đặt đường dẫn để import các module trong process_data
sys.path.append('..') 
# Import trực tiếp từ file preprocess.py
from process_data.preprocess import process_json_chunked, preprocess_data
from process_data.feature_engineering import engineer_features


# 2. Load dữ liệu raw (Chọn đúng file test_v2.pkl hoặc file bạn đang cần xử lý)
DATA_PATH = '../data/test_v2.pkl'
print(f"--- Đang load dữ liệu từ {DATA_PATH} ---")
df = pd.read_pickle(DATA_PATH)


--- Đang load dữ liệu từ ../data/test_v2.pkl ---


In [5]:
print("\n--- Bước 1: Giải nén JSON (Chunking) ---")
df = process_json_chunked(df)


--- Bước 1: Giải nén JSON (Chunking) ---

🚀 PHASE 1: JSON PARSING & FLATTENING
   401,589 dòng | 9 chunks

   Chunk 1/9
   Chunk 2/9
   Chunk 3/9
   Chunk 4/9
   Chunk 5/9
   Chunk 6/9
   Chunk 7/9
   Chunk 8/9
   Chunk 9/9
   ✅ JSON parsing hoàn tất!



In [6]:
# 4. Bước 2: Preprocessing (Dọn dẹp, chuẩn hóa số liệu và ngày tháng)
print("\n--- Bước 2: Preprocessing (Dọn dẹp & Chuẩn hóa) ---")
df = preprocess_data(df)


--- Bước 2: Preprocessing (Dọn dẹp & Chuẩn hóa) ---

📊 PHASE 2: DATA CLEANING & PREPROCESSING

   > Cleaning revenue...
   > Detecting & removing junk...
      Loại bỏ 10,432 sessions (2.60%)
      Còn lại: 391,157 sessions
   > Dropping low-signal columns...
      Dropped: ['trafficSource_adwordsClickInfo.gclId', 'geoNetwork_networkDomain', 'trafficSource_keyword', 'trafficSource_referralPath', 'trafficSource_adContent', 'trafficSource_adwordsClickInfo.slot', 'trafficSource_adwordsClickInfo.adNetworkType']
   > Filling missing values...
   > Normalizing traffic source...
   > Adding time features...
   > Adding network_type...

   ✅ Preprocessing hoàn tất!



In [7]:
# 5. Load artifacts
print("\n--- Bước 3: Load artifacts từ Train ---")
with open('../data/fe_artifacts.pkl', 'rb') as f:
    artifacts = pickle.load(f)
# 6. Tính toán Feature Window
feature_end_date    = df['date'].max()
feature_window_days = (feature_end_date - df['date'].min()).days
print(f"Feature window: {feature_window_days} ngày")



--- Bước 3: Load artifacts từ Train ---


Feature window: 167 ngày


In [8]:

# 7. Feature Engineering (User-level aggregation)
print("\n--- Bước 4: Feature Engineering ---")
X_test = engineer_features(
    df_feat             = df.copy(),
    feature_end_date    = feature_end_date,
    artifacts           = artifacts,
    feature_window_days = feature_window_days,
)


--- Bước 4: Feature Engineering ---


In [9]:
# 8. Lưu kết quả
OUTPUT_PATH = '../data/real_test_final.pkl'
X_test.to_pickle(OUTPUT_PATH)
print(f"\n✅ HOÀN THÀNH! File lưu tại: {OUTPUT_PATH}")


✅ HOÀN THÀNH! File lưu tại: ../data/real_test_final.pkl


In [10]:
print(f"  - Shape dữ liệu sau FE: {X_test.shape}")
print(f"  - File đã lưu tại: {OUTPUT_PATH}")

  - Shape dữ liệu sau FE: (290013, 67)
  - File đã lưu tại: ../data/real_test_final.pkl


In [11]:
X_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 290013 entries, 0 to 290012
Data columns (total 67 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   fullVisitorId                     290013 non-null  str    
 1   visitId_count                     290013 non-null  int64  
 2   visitNumber_max                   290013 non-null  int64  
 3   totals_hits_sum                   290013 non-null  int64  
 4   funnel_depth                      290013 non-null  float64
 5   totals_pageviews_sum              290013 non-null  float64
 6   totals_pageviews_mean             290013 non-null  float64
 7   totals_timeOnSite_sum             290013 non-null  float64
 8   totals_timeOnSite_mean            290013 non-null  float64
 9   totals_bounces_mean               290013 non-null  float64
 10  totals_sessionQualityDim_mean     290013 non-null  float64
 11  totals_sessionQualityDim_max      290013 non-null  int64  
 12 